# 01: Veri Seti Tanıtımı ve Ön İşleme (Kişi 1)

**Proje:** Framingham Kalp Çalışması verisi ile inme (stroke) tahmini

**Bu notebook'ta yapılanlar:**
1. Veri setinin yüklenmesi ve tanıtımı
2. Problemin tanımlanması: hedef değişken, analize alınan kişiler
3. Veri sızıntısına yol açan sütunların çıkarılması
4. Eksik ve tutarsız verilerin tespiti
5. Anormal (aykırı) değerlerin incelenmesi
6. Eğitim/test ayrımı ve eksik verilerin doldurulması
7. Temiz verinin kaydedilmesi (`train.csv`, `test.csv`)

**Veri seti:** https://www.kaggle.com/datasets/shreyjain601/framingham-heart-study
(NHLBI Framingham Heart Study eğitim amaçlı boylamsal veri seti)

## 1. Kütüphaneler ve sabitler

Ortak sabitler (`RANDOM_STATE`, `TARGET`, `TEST_SIZE`) ve veri klasörü (`DATA_DIR`) `src/config.py` dosyasından gelir. Colab'da `config.py` dosyasını sol menüdeki klasör simgesinden yükleyin.

In [ ]:
import glob
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split

# Repo: ../src. Colab: the src/*.py files uploaded to the working directory.
SRC_DIR = next((p for p in (Path("../src"), Path("src"), Path(".")) if (p / "config.py").exists()), None)
if SRC_DIR is None:
    raise FileNotFoundError("config.py bulunamadı. Colab'da src klasöründeki .py dosyalarını sol menüden yükleyin.")
sys.path.insert(0, str(SRC_DIR.resolve()))

from config import DATA_DIR, RANDOM_STATE, TARGET, TEST_SIZE

pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")

## 2. Veri setinin yüklenmesi

In [ ]:
import kagglehub
dataset_dir = kagglehub.dataset_download("shreyjain601/framingham-heart-study")
csv_path = glob.glob(os.path.join(dataset_dir, "**", "*.csv"), recursive=True)[0]
df_raw = pd.read_csv(csv_path)

print("Dosya:", os.path.basename(csv_path))
print("Boyut (satır, sütun):", df_raw.shape)
df_raw.head()

## 3. Veri setinin tanıtımı

Framingham Kalp Çalışması 1948'de ABD'nin Framingham kasabasında başlamış, kalp ve damar hastalıklarının risk faktörlerini inceleyen uzun süreli bir çalışmadır. Bu veri seti, 4.434 katılımcının 3 farklı muayene dönemindeki (PERIOD = 1, 2, 3) ölçümlerini ve yaklaşık 24 yıllık takip süresinde yaşadıkları olayları (inme, kalp krizi, ölüm vb.) içerir.

**Veri setindeki sütun grupları:**

| Grup | Sütunlar | Açıklama |
|---|---|---|
| Kimlik / zaman | `RANDID`, `PERIOD`, `TIME` | Kişi numarası, muayene dönemi, başlangıçtan bu yana geçen gün |
| Demografik | `SEX`, `AGE`, `educ` | Cinsiyet (1 = erkek, 2 = kadın), yaş, eğitim düzeyi (1-4) |
| Klinik ölçümler | `TOTCHOL`, `SYSBP`, `DIABP`, `BMI`, `HEARTRTE`, `GLUCOSE`, `HDLC`, `LDLC` | Kolesterol, büyük/küçük tansiyon, vücut kitle indeksi, nabız, glikoz |
| Yaşam tarzı / ilaç | `CURSMOKE`, `CIGPDAY`, `BPMEDS` | Sigara içme, günlük sigara sayısı, tansiyon ilacı |
| Muayene anındaki hastalıklar | `DIABETES`, `PREVCHD`, `PREVAP`, `PREVMI`, `PREVSTRK`, `PREVHYP` | Diyabet, koroner kalp hastalığı, anjina, kalp krizi, inme, hipertansiyon öyküsü |
| Takip sonuçları | `DEATH`, `ANGINA`, `HOSPMI`, `MI_FCHD`, `ANYCHD`, `STROKE`, `CVD`, `HYPERTEN` | Takip süresince olay yaşandı mı (0/1) |
| Olay zamanları | `TIMEAP`, `TIMEMI`, `TIMEMIFC`, `TIMECHD`, `TIMESTRK`, `TIMECVD`, `TIMEDTH`, `TIMEHYP` | Olaya kadar geçen gün sayısı |

In [ ]:
df_raw.info()

In [ ]:
print("Benzersiz katılımcı sayısı:", df_raw["RANDID"].nunique())
print("\nMuayene dönemlerine göre satır sayısı:")
print(df_raw["PERIOD"].value_counts().sort_index())

## 4. Problemin tanımlanması

**Hedef değişken:** `STROKE` (takip süresince inme geçirdi mi? 0 = hayır, 1 = evet)

Veri boylamsal olduğu için aynı kişi 3 satıra kadar tekrar edebilir. Bunu olduğu gibi kullanırsak aynı kişinin bir satırı eğitim, diğer satırı test verisine düşer ve model sonuçları gerçekte olduğundan iyi görünür. Bu yüzden:

1. **Sadece ilk muayene (PERIOD = 1) alınır.** Her kişi tek satırla temsil edilir ve tahmin, takibin başındaki bilgilerle yapılır.
2. **Muayene anında zaten inme geçirmiş kişiler (PREVSTRK = 1) çıkarılır.** Amaç yeni bir inmeyi önceden tahmin etmektir.

In [ ]:
df = df_raw[df_raw["PERIOD"] == 1].copy()
print("PERIOD = 1 sonrası:", df.shape)

n_prev_stroke = (df["PREVSTRK"] == 1).sum()
df = df[df["PREVSTRK"] == 0].copy()
print(f"Daha önce inme geçirmiş {n_prev_stroke} kişi çıkarıldı. Kalan: {df.shape}")

## 5. Veri sızıntısına yol açan sütunların çıkarılması

Bazı sütunlar ancak takip süresi boyunca ortaya çıkan bilgileri içerir. Gerçek hayatta tahmin anında bu bilgiler bilinemez. Model bunları görürse "geleceği bilerek" tahmin yapar ve sonuçlar yapay olarak yüksek çıkar (veri sızıntısı / data leakage).

| Çıkarılan sütunlar | Neden |
|---|---|
| `RANDID` | Kimlik numarası, bilgi taşımaz |
| `PERIOD`, `TIME` | Tüm satırlarda sabit (ilk muayene) |
| `PREVSTRK` | Filtrelemeden sonra hep 0 |
| `TIMESTRK` ve diğer tüm `TIME...` sütunları | Olaya kadar geçen süre: hedefi doğrudan ele verir |
| `DEATH`, `ANGINA`, `HOSPMI`, `MI_FCHD`, `ANYCHD`, `CVD`, `HYPERTEN` | Takip süresince oluşan diğer sonuçlar |

In [ ]:
leakage_cols = ["RANDID", "PERIOD", "TIME", "PREVSTRK",
                "DEATH", "ANGINA", "HOSPMI", "MI_FCHD", "ANYCHD", "CVD", "HYPERTEN"]
leakage_cols += [c for c in df.columns if c.startswith("TIME")]
leakage_cols = sorted(set(c for c in leakage_cols if c in df.columns))

df = df.drop(columns=leakage_cols)
print("Çıkarılan sütunlar:", leakage_cols)
print("Kalan boyut:", df.shape)
print("Kalan sütunlar:", list(df.columns))

## 6. Hedef değişkenin dağılımı

In [ ]:
counts = df[TARGET].value_counts().sort_index()
percent = df[TARGET].value_counts(normalize=True).sort_index() * 100
print(pd.DataFrame({"count": counts, "percent": percent.round(2)}))

ax = counts.plot(kind="bar", color=["#4C72B0", "#C44E52"])
ax.set_xticklabels(["İnme yok (0)", "İnme var (1)"], rotation=0)
ax.set_title("Hedef değişken dağılımı (STROKE)")
ax.set_ylabel("Kişi sayısı")
plt.show()

Sınıflar dengesizdir: inme geçiren kişiler azınlıktadır. Bu nedenle model değerlendirmesinde accuracy yerine recall, F1 ve ROC-AUC kullanılacak, dengeleme (SMOTE) ise sadece eğitim verisine uygulanacaktır (02 numaralı notebook).

## 7. Eksik verilerin tespiti

In [ ]:
missing = df.isnull().sum()
missing_table = pd.DataFrame({"missing": missing, "percent": (missing / len(df) * 100).round(2)})
missing_table = missing_table[missing_table["missing"] > 0].sort_values("missing", ascending=False)
print(missing_table)

missing_table["percent"].plot(kind="barh", color="#DD8452")
plt.title("Sütunlara göre eksik veri yüzdesi")
plt.xlabel("% eksik")
plt.gca().invert_yaxis()
plt.show()

**Yorum:** `HDLC` ve `LDLC` (iyi ve kötü kolesterol) yalnızca 3. muayene döneminde ölçülmüştür. İlk muayenede tamamen boş oldukları için doldurulamazlar ve çıkarılırlar. Diğer sütunlardaki eksikler ise az sayıdadır ve doldurulacaktır (bkz. Bölüm 10).

In [ ]:
high_missing = [c for c in df.columns if df[c].isnull().mean() > 0.5]
df = df.drop(columns=high_missing)
print("%50'den fazlası boş olduğu için çıkarılan sütunlar:", high_missing)
print("Kalan boyut:", df.shape)

## 8. Tekrar eden ve tutarsız kayıtlar

In [ ]:
print("Tekrar eden satır sayısı:", df.duplicated().sum())

nonsmoker_with_cigs = df[(df["CURSMOKE"] == 0) & (df["CIGPDAY"] > 0)]
smoker_without_cigs = df[(df["CURSMOKE"] == 1) & (df["CIGPDAY"] == 0)]
print("Sigara içmiyor ama günlük sigara > 0:", len(nonsmoker_with_cigs))
print("Sigara içiyor ama günlük sigara = 0:", len(smoker_without_cigs))
print("Sigara içmeyen ama CIGPDAY'i boş olan:", (df["CIGPDAY"].isnull() & (df["CURSMOKE"] == 0)).sum())
print("Sigara içen ve CIGPDAY'i boş olan:", (df["CIGPDAY"].isnull() & (df["CURSMOKE"] == 1)).sum())
print("Küçük tansiyon >= büyük tansiyon:", (df["DIABP"] >= df["SYSBP"]).sum())

Sigara içip günlük sigara sayısı 0 olan kişiler mantıksal bir tutarsızlıktır. Bu değerler eksik veri (NaN) olarak işaretlenir ve aşağıda sigara içenlerin medyanı ile doldurulur.

In [ ]:
df.loc[(df["CURSMOKE"] == 1) & (df["CIGPDAY"] == 0), "CIGPDAY"] = np.nan
df.loc[(df["CURSMOKE"] == 0) & (df["CIGPDAY"].isnull()), "CIGPDAY"] = 0  # non-smoker: certainly 0
print("CIGPDAY eksik sayısı (düzeltme sonrası):", df["CIGPDAY"].isnull().sum())

## 9. Anormal (aykırı) değerlerin incelenmesi

In [ ]:
continuous = [c for c in ["AGE", "TOTCHOL", "SYSBP", "DIABP", "BMI", "HEARTRTE", "GLUCOSE", "CIGPDAY"] if c in df.columns]

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, c in zip(axes.ravel(), continuous):
    sns.boxplot(y=df[c], ax=ax, color="#8DA0CB")
    ax.set_title(c)
plt.suptitle("Sürekli değişkenlerde aykırı değerler (boxplot)")
plt.tight_layout()
plt.show()

rows = []
for c in continuous:
    q1, q3 = df[c].quantile([0.25, 0.75])
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    rows.append([c, round(df[c].min(), 1), round(df[c].max(), 1), ((df[c] < low) | (df[c] > high)).sum()])
print(pd.DataFrame(rows, columns=["column", "min", "max", "iqr_outliers"]).to_string(index=False))

**Klinik olarak mümkün olmayan değer kontrolü:** IQR yöntemi pek çok değeri aykırı olarak işaretler, ancak tıbbi verilerde yüksek tansiyon, yüksek glikoz ya da yüksek kolesterol gerçek ve **önemli** bilgilerdir (inme riskini doğrudan etkiler). Bu nedenle aykırı değerler silinmez. Yalnızca insan fizyolojisine göre imkânsız olan değerler (ölçüm/kayıt hatası) kaldırılır.

In [ ]:
clinical_limits = {  # (min, max) plausible value
    "SYSBP":    (70, 300),
    "DIABP":    (30, 200),
    "BMI":      (12, 70),
    "HEARTRTE": (30, 220),
    "GLUCOSE":  (30, 600),
    "TOTCHOL":  (80, 700),
}
invalid = pd.Series(False, index=df.index)
for c, (low, high) in clinical_limits.items():
    if c in df.columns:
        mask = df[c].notnull() & ((df[c] < low) | (df[c] > high))
        print(f"{c}: {mask.sum()} imkânsız değer")
        invalid |= mask

bp_error = df["DIABP"] >= df["SYSBP"]
print(f"Küçük tansiyon >= büyük tansiyon: {bp_error.sum()} kayıt hatası")
invalid |= bp_error

df = df[~invalid].copy()
print("\nKlinik kontrol sonrası boyut:", df.shape)

## 10. Eğitim/test ayrımı ve eksik verilerin doldurulması

Eksik değerleri doldurmak için kullanılan istatistikler (medyan, mod) **yalnızca eğitim verisinden** hesaplanır ve test verisine aynen uygulanır. Böylece test verisindeki bilgiler eğitim sürecine sızmaz.

- **Ayırma yöntemi:** Hold-out, %80 eğitim / %20 test, hedefe göre katmanlı (stratified), `random_state = 42`
- **Sürekli değişkenler** (`TOTCHOL`, `BMI`, `HEARTRTE`, `GLUCOSE`): **medyan**. Aykırı değerlerden etkilenmez.
- **`CIGPDAY`**: sigara içenlerin medyanı (içmeyenler zaten 0 yapıldı)
- **Kategorik / ikili** (`educ`, `BPMEDS`): **mod** (en sık değer)

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE)

print("Eğitim:", X_train.shape, "| İnme oranı: %.2f%%" % (y_train.mean() * 100))
print("Test:  ", X_test.shape, "| İnme oranı: %.2f%%" % (y_test.mean() * 100))

In [ ]:
X_train = X_train.copy()
X_test = X_test.copy()

median_cols = [c for c in ["TOTCHOL", "BMI", "HEARTRTE", "GLUCOSE"] if c in X_train.columns]
mode_cols = [c for c in ["educ", "BPMEDS"] if c in X_train.columns]

fill_values = {}
for c in median_cols:
    fill_values[c] = X_train[c].median()
for c in mode_cols:
    fill_values[c] = X_train[c].mode()[0]
fill_values["CIGPDAY"] = X_train.loc[X_train["CURSMOKE"] == 1, "CIGPDAY"].median()

X_train = X_train.fillna(fill_values)
X_test = X_test.fillna(fill_values)

print("Doldurma değerleri (eğitim verisinden):")
for k, v in fill_values.items():
    print(f"  {k}: {v}")
print("\nKalan eksik (eğitim):", X_train.isnull().sum().sum(), "| (test):", X_test.isnull().sum().sum())

## 11. Son durum ve kaydetme

In [ ]:
print("Son özellik sayısı:", X_train.shape[1])
print("Özellikler:", list(X_train.columns))
X_train.describe().T.round(2)

In [ ]:
train = X_train.assign(**{TARGET: y_train})
test = X_test.assign(**{TARGET: y_test})
DATA_DIR.mkdir(exist_ok=True)
train.to_csv(DATA_DIR / "train.csv", index=False)
test.to_csv(DATA_DIR / "test.csv", index=False)
print(f"Kaydedildi: {DATA_DIR / 'train.csv'} {train.shape} | {DATA_DIR / 'test.csv'} {test.shape}")

In [ ]:
# Colab: download the files to share them with the team (or copy them to Drive)
try:
    from google.colab import files
    files.download(str(DATA_DIR / "train.csv"))
    files.download(str(DATA_DIR / "test.csv"))
except ImportError:
    print(f"Colab dışında çalışıyor, dosyalar {DATA_DIR} klasörüne kaydedildi.")

## 12. Özet (rapor için)

- Ham veri: 11.627 satır, 39 sütun (4.434 katılımcının 3 muayene dönemi).
- Her katılımcının yalnızca ilk muayenesi alındı, muayene anında inme öyküsü olanlar çıkarıldı.
- Takip süresince oluşan bilgileri içeren sütunlar veri sızıntısını önlemek için çıkarıldı.
- İlk muayenede ölçülmemiş `HDLC` ve `LDLC` sütunları çıkarıldı.
- Sigara verisindeki mantıksal tutarsızlıklar düzeltildi.
- Aykırı değerler klinik olarak anlamlı olduğu için korundu, yalnızca fizyolojik olarak imkânsız değerler kaldırıldı.
- Veri %80/%20 katmanlı olarak ayrıldı, eksik değerler yalnızca eğitim verisinden hesaplanan medyan/mod ile dolduruldu.
- Çıktılar: `train.csv` ve `test.csv` (repo'da `data/` klasörü, Colab'da çalışma klasörü; Kişi 2 bu dosyalarla encoding, ölçekleme ve SMOTE adımlarına devam eder).